# PGx PageRank – הורדת נתונים
הורדת נתוני ClinPGx (לשעבר PharmGKB) לבניית רשת תרופה → אלל → גן.

| קובץ | תוכן | שימוש ברשת |
|---|---|---|
| `clinicalAnnotations.zip` | וריאנט–גן–תרופה + Level of Evidence + Score | קשתות תרופה–אלל (משקל), אלל–גן |
| `variantAnnotations.zip` | אסוציאציות ברמת מאמר, כולל כיוון האפקט | משקל לפי אפקט / מספר מאמרים |
| `drugs.zip`, `genes.zip` | מטא-דאטה על תרופות וגנים | מיפוי שמות ומזהים |

רישיון: CC BY-SA 4.0 – לציין את ClinPGx/PharmGKB כמקור.

In [ ]:
from pathlib import Path
import zipfile, requests
import pandas as pd

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# ClinPGx הוא הבית החדש של PharmGKB; ה-URL הישן נשאר כגיבוי
BASES = [
    "https://api.clinpgx.org/v1/download/file/data/",
    "https://api.pharmgkb.org/v1/download/file/data/",
]
FILES = ["clinicalAnnotations.zip", "variantAnnotations.zip", "drugs.zip", "genes.zip"]

In [ ]:
def download(fname: str) -> Path:
    out = DATA_DIR / fname
    if out.exists():
        print(f"✓ {fname} כבר קיים")
        return out
    for base in BASES:
        try:
            r = requests.get(base + fname, timeout=120)
            r.raise_for_status()
            out.write_bytes(r.content)
            print(f"✓ {fname} ({len(r.content)/1e6:.1f} MB) from {base}")
            return out
        except Exception as e:
            print(f"✗ {base}{fname}: {e}")
    raise RuntimeError(f"Download failed for {fname} – download manually from https://www.clinpgx.org/downloads into data/")

for f in FILES:
    z = download(f)
    target = DATA_DIR / z.stem
    if not target.exists():
        with zipfile.ZipFile(z) as zf:
            zf.extractall(target)
    print("  ", sorted(p.name for p in target.iterdir()))

## טעינה ובדיקה ראשונית

In [ ]:
ca = pd.read_csv(DATA_DIR / "clinicalAnnotations" / "clinical_annotations.tsv", sep="\t")
print(ca.shape)
print(ca.columns.tolist())
ca.head()

In [ ]:
# התפלגות רמות הראיות – בסיס למשקלי הקשתות תרופה–אלל
ca["Level of Evidence"].value_counts().sort_index()

In [ ]:
va = pd.read_csv(DATA_DIR / "variantAnnotations" / "var_drug_ann.tsv", sep="\t")
print(va.shape)
print(va.columns.tolist())
va.head()

In [ ]:
# סניטי: מה ידוע על קלופידוגרל? (מצופה: CYP2C19 בולט)
drug_col = next(c for c in ca.columns if c.lower().startswith("drug") or c.lower().startswith("chemical"))
ca[ca[drug_col].str.contains("clopidogrel", case=False, na=False)].head(15)

## המשך (לעבודה שלך)
1. לבחור משקל לקשת תרופה–אלל (מיפוי Level → מספר, או עמודת Score).
2. לבנות גרף (`networkx`) עם צמתים מסוג drug / variant / gene.
3. Personalized PageRank עם `personalization={drug: 1}` ולדרג רק צמתי גן.
4. בדיקות: clopidogrel→CYP2C19, warfarin→CYP2C9/VKORC1; השפעת alpha; הטיה לגנים עם degree גבוה.